In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import json
import timeit
import pickle
from multiprocessing import Pool
import os
import pandas as pd
from functools import partial
from constants import YEAR, ESPN_GROUP_FILE, ESPN_OUTPUT_STUB, ESPN_SCORES
from joblib import Parallel, delayed

## Scrape Leaderboard

In [3]:
GENDER = "mens"

In [10]:
YEAR = 2024
EXTRA = ''
if GENDER == "womens":
    EXTRA = '-women'

API_BASE = (
    "https://gambit-api.fantasy.espn.com/apis/v1/challenges/"
    f"tournament-challenge-bracket{EXTRA}-{YEAR}/leaderboard?view=pagetype_leaderboard"
)
HEADERS = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    'Gambit-Filter': '{"limit": 200, "offset": 0}'
}

In [11]:
res = requests.get(API_BASE, headers=HEADERS)
scores = [r['score']['overallScore'] for r in res.json()['entries']]
len(scores)

1

In [12]:
scores[:5]

[1680]

In [9]:
scores[-5:]

[1680, 1680, 1680, 1680, 1680]

In [7]:
with open(ESPN_SCORES.format(gender=GENDER), 'w') as f:
    json.dump(scores, f)

# This doesn't really work anymore :(

In [3]:
GENDER = "mens"

In [4]:
API_BASE = (
    f"https://fantasy.espncdn.com/tournament-challenge-bracket/{YEAR}/en/api/"
    if GENDER == "mens" else
    f"https://fantasy.espncdn.com/tournament-challenge-bracket-women/{YEAR}/en/api/"
)

FIND_GROUP = "findGroups?type_access=0&type_group=0&type_lock=0&type_dropWorst=0&start={start}&num={length}&search="

SAVE_GAP = 10000

In [23]:
API_BASE = "https://gambit-api.fantasy.espn.com/apis/v1/challenges/"
HEADERS = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    'Gambit-Filter': '{"limit": 10000, "offset": 0}'
}

In [24]:
res = requests.get(API_BASE + "240/groups?platform=chui&view=chui_default_group", headers=HEADERS)

In [25]:
len(res.json())

10000

In [28]:
res.json()[0]

{'challengeId': 240,
 'challengeSettings': {'entriesPerMember': 25,
  'scoringFormatId': 5,
  'scoringSettings': {'excludeLowScoringPeriodScore': False,
   'ignoreTiebreak': False}},
 'featuredEntryIds': ['2d08eae0-e62d-11ee-89c1-3d1e744a9e52',
  'b6a069c0-e2dd-11ee-92f6-b7591be3f41e',
  '99a5d760-e2d8-11ee-a52a-d926bb67fa32',
  'f1d9f330-e2d8-11ee-b440-d574b8f106cc',
  'b5c80090-e2dc-11ee-a6ab-ade7d1c25cad',
  '781ba6c0-e2ff-11ee-8e90-154d0ba94b5b',
  '7b09b3e0-e2dc-11ee-a52a-d926bb67fa32',
  'f02f2960-e549-11ee-a868-e93723f5b466',
  'b343a420-e654-11ee-ad97-179bbde3bbd7',
  '25d0e470-e552-11ee-b2a2-d165ed155245',
  'd8cdf250-e603-11ee-94f2-498fe42235ee',
  '0d0d7800-e58d-11ee-81c4-7bef04977ba4',
  'ffe3be80-e5de-11ee-b4a0-396cd6974492'],
 'forecastEligible': False,
 'groupId': '6e682872-7e5f-3aa2-84bf-003cb6a630ae',
 'groupSettings': {'adminMembers': [],
  'creatorMember': {'displayName': 'Chris Basten',
   'id': '{D057F0A5-121E-4100-BE61-95D1B05B25B1}'},
  'creatorMemberId': '{D057F

In [29]:
groups = [r["groupId"] for r in res.json()]

In [32]:
groups[:5]

['6e682872-7e5f-3aa2-84bf-003cb6a630ae',
 'a12288ff-0cca-4c5d-9ee6-6c4c169775d2',
 '165e0696-4839-41e7-9bae-b7c29b4b7f57',
 '2310a549-4a6f-42d2-9230-c848ca20eba0',
 '9943b705-b3aa-4c26-93f2-168a54fd56c9']

In [129]:
HEADERS = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
#     'Gambit-Filter': '{"filterSortId":{"value":null}, "limit": 200, "offset": 98}'
}
res = requests.get(API_BASE + f"tournament-challenge-bracket-2024/leaderboard?view=pagetype_leaderboard", headers=HEADERS)

In [131]:
[r['name'] for r in res.json()['entries']]

['Drew Baiera picks',
 'Let’s Go Boilers',
 'JP39',
 "Bear's Picks 2",
 "pfunkmiller's Picks 1",
 'Why Us?',
 "EDETTEWINNING's Picks 23",
 "ESPNfan4578375622's Picks 4",
 "Rasurejumper36's Picks 19",
 "ESPNFAN4450915943's Picks 1",
 'Voldemort',
 'marie',
 'Armando Bacot is 24',
 'Will Meyers 5',
 '\U0001faf8🏀',
 'Return of the Mac',
 'Nick Fraulino 1',
 'clyxfrd',
 'Milo’s bracket🏀',
 "ESPNFAN0353258269's Picks 1",
 "Flyrink\r's Picks 25",
 'Post Up Fencing IL',
 'Alieu',
 "ESPNFAN05954710's Picks 1",
 "ESPNFAN9158084958's Picks 1",
 'Alena Shores - Bracket',
 '🍗',
 "dddhouse's Picks 1",
 'tee hee',
 'Hal Melstad',
 'Lyndsey',
 "ESPNFAN9981845278's Picks 1",
 "FUNROWER's Picks 5",
 'Feldy',
 ' ¯\\_(ツ)_/¯',
 "ESPNFAN0944891033's Picks 7",
 "CLEATSSHOWER's Picks 16",
 "Caiaphas427's Picks 22",
 "ESPNFAN8788872034's Picks 3",
 'Linc’s picks   ',
 'Ashtyn’s Picks',
 'cade dillavou',
 'Jimmybobby',
 'GirlDadMD',
 "Keifer Ewing's Picks 2",
 "QueenBeze's Picks 1",
 "espn07587083's Picks 1",


In [132]:
scores = [r['score']['overallScore'] for r in res.json()['entries']]

In [133]:
len(scores)

50

In [134]:
scores

[610,
 610,
 610,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 600,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590,
 590]

In [99]:
%%time
scores = []
for group in groups[:100]:
    res = requests.get(API_BASE + f"tournament-challenge-bracket-2024/groups/{group}", headers=HEADERS)
    scores.extend([r['score']['overallScore'] for r in res.json()['entries']])

CPU times: user 9.64 s, sys: 983 ms, total: 10.6 s
Wall time: 1min 8s


In [82]:
res.json()
[r['name'] for r in res.json()['entries']]

["JoeBog1994's Picks 1"]

In [5]:
def get_espn_groups(length=100000):
    groups = []
    start = 0
    start_time = timeit.default_timer()
    while True:
        resp = requests.get(API_BASE + FIND_GROUP.format(start=start, length=length))
        data = json.loads(resp.text)["g"]
        if len(data) > 0:
            groups.extend([group["id"] for group in data])
            stop_time = timeit.default_timer()
            print(f"{start}: {stop_time - start_time}")
            start += length
        else:
            break
    return groups

In [6]:
%%time
groups = get_espn_groups()

JSONDecodeError: Expecting value: line 2 column 3 (char 3)

In [23]:
len(groups)

110320

In [24]:
groups = sorted(list(set(groups)))

In [25]:
pickle.dump(groups, open(ESPN_GROUP_FILE.format(gender=GENDER), 'wb'))

In [47]:
def by_group(group_id):
    url = API_BASE + 'group?length=10000&groupID=' + str(group_id)
    d = {}
    start = 0
    while True:
        try:
            resp = requests.get(url + '&start=' + str(start))
        except requests.exceptions.RequestException as e:
            print("{}: {}".format(group_id, e))
            continue
        try:
            data = json.loads(resp.text)
            if 'g' not in data or 'e' not in data['g']:
                break
            else:
                ent = data['g']['e']
                start += len(ent)
                for e in ent:
                    d[e['id']] = e
        except Exception as e:
            print("{}: {}".format(group_id, e))
            continue
    return d


def groups_from_list(group_file, save_file_stub, save_gap, start=0, i=0):
    start_time = timeit.default_timer()

    groups = pickle.load(open(group_file, "rb"))

    for start in range(start, len(groups), save_gap):
        entries = {}
        stop = start + save_gap
        stop = len(groups) if stop > len(groups) else stop
        for group in groups[start:stop]:
            entries.update(by_group(group))
        with open(save_file_stub.format(i), 'w') as f:
            json.dump(entries, f)
        stop_time = timeit.default_timer()
        print(f"{stop}: {stop_time - start_time}")
        i += 1
        
        
def parallel_scrape(group_file, save_file_stub, save_gap, start=0, i=0, threads=4):
    start_time = timeit.default_timer()

    groups = pickle.load(open(group_file, "rb"))

    for start in range(start, len(groups), save_gap):
        entries = {}
        stop = start + save_gap
        stop = len(groups) if stop > len(groups) else stop
        
        res = Parallel(threads)(delayed(by_group)(group_id) for group_id in groups[start:stop])
        entries = {
            key: value
            for r in res
            for key, value in r.items()
        }
        with open(save_file_stub.format(i), 'w') as f:
            json.dump(entries, f)
        stop_time = timeit.default_timer()
        print(f"{stop}: {stop_time - start_time}")
        i += 1

## Make sure the directories exist first!!

In [43]:
%%time

# COLD START
# start = 0
# i = 0

# RESTART
start = SAVE_GAP
i = 1

groups_from_list(
    group_file=ESPN_GROUP_FILE.format(gender=GENDER),
    save_file_stub=ESPN_OUTPUT_STUB.format(gender=GENDER),
    save_gap=SAVE_GAP,
    start=start,
    i=i,
)

20000: 1663.4279001069954
30000: 3317.8592470980075
40000: 5006.210160626011
50000: 6558.468105770007
60000: 9108.890671998
70000: 12055.281774783012
80000: 14864.881037832005
90000: 17885.426861432003
162064: Expecting value: line 1 column 1 (char 0)
100000: 21581.86721756999
164171: Expecting value: line 1 column 1 (char 0)
164709: Expecting value: line 1 column 1 (char 0)
110000: 24711.481977841002
110317: 24843.467837122007
CPU times: user 1h 4min 26s, sys: 3min 38s, total: 1h 8min 5s
Wall time: 6h 54min 5s


In [51]:
%%time

# COLD START
start = 0
i = 0

# RESTART
# start = SAVE_GAP
# i = 1

parallel_scrape(
    group_file=ESPN_GROUP_FILE.format(gender=GENDER),
    save_file_stub=ESPN_OUTPUT_STUB.format(gender=GENDER),
    save_gap=SAVE_GAP,
    start=start,
    i=i,
)

10000: 766.8827885699866
20000: 1372.1650937729864
30000: 1969.1755262969818
40000: 2554.0188814299763
50000: 3112.7716767620004
60000: 3639.691664861981
70000: 4140.952149999997
80000: 4573.927230707981
90000: 5092.738698647998
100000: 5550.09222089799
110000: 6041.880768317002


/Users/Alex/opt/anaconda3/envs/py3/lib/python3.8/site-packages/joblib/externals/loky/process_executor.py:703: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


120000: 6518.639007602993
130000: 6981.513666332001
140000: 7440.759792834986
150000: 7900.99128895998
160000: 8489.516099392
170000: 9007.598447333992
180000: 9467.17783862399
190000: 9927.829479275999
200000: 10385.803340990999
210000: 10846.36774687699
220000: 11302.311571833998
230000: 11727.288019486994
240000: 12157.433444998984
250000: 12593.985080571001
260000: 13036.06884352598
270000: 13469.066273140983
280000: 13900.486034234986
290000: 14326.907234668004
300000: 14769.445005168003
310000: 15194.543980196002
320000: 15622.887677992985
330000: 16062.121332045994
340000: 16492.940487236978
350000: 16919.003888665
360000: 17335.770006765
370000: 17750.477398522984
380000: 18178.59516934
390000: 18605.972211950982
400000: 19037.312833835982
410000: 19459.274547754
420000: 19867.230293029977
430000: 20282.196004063997
440000: 20687.859904746
450000: 21085.59717632699
460000: 21491.974753635994
470000: 21893.16137113198
480000: 22314.990817299986
490000: 22705.313834470988
500000: